# user websockets
> Solveit compatible user-defined HTMX websockets.

In [ ]:
#| default_exp user_ws

In [ ]:
#| hide
import json
from urllib.parse import parse_qsl, urlencode, urlparse, urlunparse
from fastcore.foundation import store_attr, AttrDict as AD
from fastcore.meta import delegates
from fasthtml.components import Div, Span
from fasthtml.core import FastHTML
from dialoghelper import iife_a, js_eval_a, add_html_a, event_get_a
from pote.basic import id_gen
from pote.dutil import wait_for
from pote.server import setup_mod, pub_domain

In [ ]:
#| hide
import asyncio
from fastcore.foundation import patch
from fastcore.net import is_port_free
from fastcore.xml import to_xml
from fastcore.test import *
from pyskills import doc
from pyskills import *
from fasthtml.components import Samp, Br
from fasthtml.xtend import Script
from dialoghelper.core import add_msg, read_msg, js_run_a, js_eval, add_scr_a
from pote.basic import simple_id
from pote.logger import logger
from pote.dutil import link_msg
from pote.server import stop_server_a, ensure_server_a, serve_app, setup_loader, add_js_msg, free_port, pub_domain, pub_ports
from htmx_bridge.core import get_hx, setup_htmx_bridge

In [ ]:
_new_id = id_gen()

In [ ]:
await setup_htmx_bridge()
hx = get_hx()

In [ ]:
await stop_server_a()
app=rt=p=None

## Problem

Solveit injects browser JavaScript that listens globally for HTMX websocket lifecycle events:

```javascript
function updateWSIndicator(type) {
  if (['error', 'close'].includes(type)) {
    htmx.find('#submit_btn').disabled = true;
    htmx.find('#cancel_btn').disabled = true;
  }
  const color = { error: "orange", open: "limegreen", close: "red" }[type] || 'grey';
  const elt = document.getElementById("ws-indicator");
  if (elt) elt.style.backgroundColor = color;
}

htmx.on("htmx:wsOpen", ({ detail: { event} }) => updateWSIndicator(event.type));
htmx.on("htmx:wsClose", ({ detail: { event } }) => updateWSIndicator(event.type));
htmx.on("htmx:wsError", () => updateWSIndicator('error'));
```

That works for Solveit’s own websocket, but it is **too broad**. If a user also uses HTMX’s websocket extension in their own app inside a dialog, their websocket emits the same `htmx:wsOpen`, `htmx:wsClose`, and `htmx:wsError` events.

So when the user’s own websocket server stops, Solveit sees `htmx:wsClose` / `htmx:wsError` and thinks **Solveit’s own backend connection** is broken. The result is:

- Solveit’s WS indicator turns red/orange
- Submit/cancel buttons may be disabled
- Console logs misleading messages like “Lost connection to solveit”
- HTMX may keep retrying the user websocket using Solveit’s global `htmx.config.wsReconnectDelay`

The real problem is not that Solveit’s websocket is down. The user’s websocket is down. Solveit is reacting to the wrong websocket events.

## Important environment detail

Inside Solveit, relative HTTP requests like:

```python
Span(hx_get='/foo', hx_target='closest div')('private')
```

can work because the Solveit backend proxies unhandled HTTP requests to `localhost:8000`.

But that proxy does **not** forward websocket upgrades. So this will not work reliably:

```html
<div hx-ext="ws" ws-connect="/ws2"></div>
```

because `/ws2` goes through Solveit’s own app/proxy path.

For user websockets, use the public `.solveit.pub` domain mapped to the user server port:

```python
ws_url = f"wss://{json.loads(os.environ['PUBLIC_DOMAINS'])[str(srv.port)]}/ws2"
```

Then inject the websocket initiator using that absolute `wss://...` URL:

```python
await add_html_a(Div(Div(ws_connect=ws_url, hx_ext='ws', id='ws2-div'), hx_swap_oob='afterend:#ephemeral'))
```

The `.solveit.pub` mapping supports websocket upgrade; verified with `101 Switching Protocols`.

## Why CORS is not the main issue

Browser WebSocket connections are not blocked by normal fetch/XHR CORS rules. The browser sends an `Origin` header, but it is up to the server to accept or reject it. If the public `.solveit.pub` tunnel forwards the upgrade correctly, the websocket should connect.

## First workaround: capture and block user websocket events

HTMX’s `htmx.on('event', fn)` attaches the listener to `document.body`, not `document`. This matters.

If we add a capturing-phase event listener on `document`, it runs before Solveit’s bubbling-phase listener on `document.body`.

So we can stop user websocket events before Solveit sees them:

```javascript
document.addEventListener('htmx:wsOpen', e => {
  if (e.target?.id !== 'ws2-div') return;
  window._myWs = e.detail.socketWrapper;
  e.stopImmediatePropagation();
  console.log('non-solveit WS opened');
}, true)

document.addEventListener('htmx:wsClose', e => {
  if (e.target?.id !== 'ws2-div') return;
  e.stopImmediatePropagation();
  console.log('blocked close from my ws');
}, true)

document.addEventListener('htmx:wsError', e => {
  if (e.target?.id !== 'ws2-div') return;
  e.stopImmediatePropagation();
  console.log('blocked error from my ws');
}, true)
```

Key details:

- Use `e.target?.id === 'ws2-div'`, not `socketWrapper` comparison.
- `htmx:wsError` uses a different wrapper shape than `htmx:wsOpen` / `htmx:wsClose`, so wrapper identity comparison is fragile.
- Target-element comparison survives reconnects, since each retry still targets the same websocket initiator element.
- This prevents Solveit’s indicator/listeners from reacting to the user websocket.

This fixes the false red/orange Solveit indicator.

## Remaining issue: HTMX still retries

Stopping propagation does not stop HTMX’s internal websocket retry logic. The retry is scheduled inside the websocket extension’s own `onclose` / `onerror` handlers before or independently of the DOM event propagation.

So even if Solveit’s UI no longer reacts, the console may still show repeated failed websocket reconnect attempts.

For intentional shutdown, the clean solution is:

```javascript
const el = document.getElementById('ws2-div')
if (el) el.remove()
```

Then stop the Python server:

```python
stop_server()
```

Removing the websocket initiator element lets HTMX clean up the websocket and avoids the retry loop. In testing, this also avoided triggering the close/error blockers at all.

So the practical shutdown sequence is:

```python
await iife_a(r'''
const el = document.getElementById('ws2-div')
if (el) el.remove()
''')

stop_server()
```

## Best overall pattern

Use both:

1. Capturing listeners as a safety net for unexpected disconnects
2. Remove the websocket initiator element before intentional server shutdown

The listeners prevent Solveit’s UI from reacting to user websocket failures. Removing the initiator avoids HTMX’s retry loop when shutdown is intentional.

## Browser → Python websocket messages

The FastHTML websocket handler expects structured JSON matching the handler parameters. Sending plain text caused a server parse error:

```javascript
window._myWs.send('world')
```

That produced:

```text
Expecting value: line 1 column 1 (char 0)
```

Correct browser send:

```javascript
window._myWs.send(JSON.stringify({msg: 'world'}))
```

Corresponding Python handler:

```python
@app.ws('/ws2', conn=on_connect, disconn=on_disconnect)
async def ws2(msg:str, send):
    _msgs.append(msg)
    if msg == '__init__': return
    await send(Div('Hello ' + msg, id='note2'))
```

## Python → browser websocket messages

FastHTML gives a `send` callable on websocket connect. Store it, then use it later from any async Python cell:

```python
_msgs = []
_ws_send = None

async def on_connect(send):
    global _ws_send
    _ws_send = send
    _msgs.append('Connected!')

async def on_disconnect(ws):
    global _ws_send
    _ws_send = None
    _msgs.append('Disconnected!')
```

Then later:

```python
await _ws_send(Div('Hello from Python', id='note2'))
```

That sends server → browser over the already-open websocket.

## Subtle timing issue with `on_connect`

Sending UI HTML from `on_connect` is unreliable.

Observed behavior:

- Server sent: `<div id="note2">Hello, you have connected</div>`
- Browser websocket inspector showed the message was received
- But HTMX did not swap it into the DOM
- Later messages, sent after explicit browser messages, swapped correctly

Adding `hx-swap-oob="true"` did not fix the first-message issue. A real `setTimeout` delay after `htmx:wsOpen` did work, suggesting a timing/settling race: the websocket is open and messages arrive, but HTMX/Solveit may not yet be ready to process the first swap.

Best practice: treat `conn=on_connect` as lifecycle bookkeeping only. Don’t send UI updates from it.

Good shape:

```python
async def on_connect(send):
    global _ws_send
    _ws_send = send
    _msgs.append('Connected!')

async def on_disconnect(ws):
    global _ws_send
    _ws_send = None
    _msgs.append('Disconnected!')

@app.ws('/ws2', conn=on_connect, disconn=on_disconnect)
async def ws2(msg:str, send):
    _msgs.append(msg)
    if msg == '__init__': return
    await send(Div('Hello ' + msg, id='note2'))
```

## Minimal complete browser-side workaround

```javascript
document.addEventListener('htmx:wsOpen', e => {
  if (e.target?.id !== 'ws2-div') return;
  window._myWs = e.detail.socketWrapper;
  e.stopImmediatePropagation();
  console.log('non-solveit WS opened');
}, true)

document.addEventListener('htmx:wsClose', e => {
  if (e.target?.id !== 'ws2-div') return;
  e.stopImmediatePropagation();
  console.log('blocked close from my ws');
}, true)

document.addEventListener('htmx:wsError', e => {
  if (e.target?.id !== 'ws2-div') return;
  e.stopImmediatePropagation();
  console.log('blocked error from my ws');
}, true)
```

Clean shutdown:

```javascript
const el = document.getElementById('ws2-div')
if (el) el.remove()
```

Browser send:

```javascript
window._myWs.send(JSON.stringify({msg: 'world'}))
```

Python send:

```python
await _ws_send(Div('Hello from Python', id='note2'))
```

## Bottom line

The Solveit bug/oversight is that its websocket monitor listens to all HTMX websocket events globally and assumes they belong to Solveit. User HTMX websockets produce the same events, so user websocket failures are misinterpreted as Solveit connection failures.

The workaround is to:

- Connect user websockets via the public `wss://...solveit.pub/...` URL, not relative paths
- Give the user websocket initiator a stable id, e.g. `ws2-div`
- Add capturing-phase document listeners to block `htmx:wsOpen`, `htmx:wsClose`, and `htmx:wsError` from that element
- Remove the websocket initiator before intentionally stopping the user server
- Avoid sending DOM updates from `on_connect`; send only after the websocket is fully settled or after an explicit client message

## version 0

In [ ]:
await setup_mod('user_ws', versioned=True)

('/esm/user_ws?v=1',
 {'data_id': 'd37dfa52-a79a-457b-ab3f-cd2cb76730e8',
  'result': {'user_ws': {}}})

In [ ]:
_ar = [Span('⬆︎', style='color:green'), Span('⬇︎', style='color:red')]
def _note(wid, msg, up=False): return Div(_ar[up], f"{wid}{': ' if wid else ''}{msg}", id='uws-note')

In [ ]:
await add_html_a(
    Div(hx_swap_oob='beforebegin:#dialog-container')(
        Div(id='uws-box', cls='flex flex-wrap w-full gap-1 lg:flex-nowrap')(
            _note('', 'Waiting...')
        )));

In [ ]:
def ws_url(path='/ws2', port=None):
    return f"wss://{pub_domain(port if port is not None else free_port(public=True))}{path}"

In [ ]:
def _indicator(wid): return Span(id=f"{wid}-uws-indicator", cls='size-4 border rounded-full min-w-4', title=wid)

In [ ]:
async def user_ws_cleanup(wid):
    res = await js_eval_a(f'''setTimeout(() => {{ $("#{wid}-uws-indicator").remove() }}, 500); return user_ws.cleanup({json.dumps(wid)})''')
    assert res.get('result') in ('removed', 'not found')

In [ ]:
async def user_ws_create_a(wid, wsurl):
    await add_html_a((
        Div(hx_swap_oob='afterend:#ws-div')(Div(ws_connect=wsurl, hx_ext='ws', id=wid, data_uws=True)),
        Div(hx_swap_oob='beforebegin:#uws-note')(_indicator(wid))))

def user_ws_create(wid, wsurl): return asyncio.create_task(user_ws_create_a(wid, wsurl))

async def user_ws_connect(wid, wsurl):
    await user_ws_create_a(wid, wsurl)
    await wait_for(lambda: _ws_send is not None)
    await wait_for(lambda: (wid, '__init__') in _msgs)

In [ ]:
srv, app, rt, p = await ensure_server_a(app=FastHTML(exts='ws', debug=True), public=True)#, log_level='info')

In [ ]:
srv.port, srv.host, ws_url('/ws2?abc=ABC', srv.port)

(8000, '0.0.0.0', 'wss://known-vent-santa-eras34.solveit.pub/ws2?abc=ABC')

In [ ]:
await wait_for(lambda: hx.eval('!!window.user_ws'));

In [ ]:
_msgs = []
_ws_send = None

async def on_connect(scope,send):
    global _ws_send
    _ws_send = send
    _msgs.append('Connected!')

async def on_disconnect(scope,ws):
    global _ws_send
    _ws_send = None
    _msgs.append('Disconnected!')

@app.ws('/ws2', conn=on_connect, disconn=on_disconnect)
async def ws2(wid:str, msg:str, send):
    _msgs.append((wid, msg))
    if msg == '__init__': return
    await send(_note(wid, msg))

In [ ]:
wid = 'ws2-div'
res = await js_eval_a(f"return user_ws.cleanup('{wid}')")
test_eq(res['result'], 'not found')

user_ws_create(wid, ws_url(f"/ws2?wid={wid}", srv.port))
await wait_for(lambda: (wid, '__init__') in _msgs)
_msgs

['Connected!', ('ws2-div', '__init__')]

Console shows:
```
{"msg":"__init__"}
```

That means the good path is working now:

- `htmx:wsOpen` fired
- our capture listener matched `ws2-div`
- it blocked Solveit from seeing the event
- `el._ws` was set
- the delayed `__init__` was sent through the HTMX socket
- Python received the structured message

Next smallest check is to verify Python saw it:

```python
_msgs
```

In [ ]:
await iife_a(f"user_ws.send('{wid}', {{msg: 'world'}})")
await wait_for(lambda: 'world' in str(_msgs[-1]))

True

In [ ]:
await user_ws_cleanup(wid)
_msgs

['Connected!', ('ws2-div', '__init__'), ('ws2-div', 'world')]

In [ ]:
_msgs.clear()
wid = 'ws3-div'
await user_ws_connect(wid, ws_url(f"/ws2?wid={wid}", srv.port))
await _ws_send(_note(wid, 'Hello from Python'))
_msgs

['Connected!', ('ws3-div', '__init__')]

websockets messages:

```
{"msg":"__init__"}	18	11:49:57.800
<div id="note2">Hello from Python</div>	40	11:59:57.474
```

I see `Hello from Python` output.

That verifies the full round trip:

- browser → Python works through `user_ws.send`
- Python receives structured JSON as `msg`
- Python → browser works through `_ws_send`
- HTMX swaps the returned `<div id="note2">...</div>` into the page

The next useful refactor target is to stop relying on a single global `_ws_send`, since multiple websocket initiators can connect to the same endpoint. A small registry keyed by websocket id / route will be more reusable.

In [ ]:
await user_ws_cleanup('ws3-div')

Next small step: replace the single `_ws_send` with a tiny registry, while keeping the current `/ws2` handler shape.

In [ ]:
_ws_sends = {}

def ws_conn(wid):
    async def _conn(send): _ws_sends[wid] = send
    return _conn

def ws_disconn(wid):
    async def _disconn(ws): _ws_sends.pop(wid, None)
    return _disconn

In [ ]:
async def user_ws_connect(wid, wsurl):
    await user_ws_create_a(wid, wsurl)
    await wait_for(lambda: wid in _ws_sends)
    await wait_for(lambda: (wid, '__init__') in _msgs)

Then a tiny helper for Python → browser sends:

In [ ]:
async def ws_send(wid, o): return await _ws_sends[wid](o)

In [ ]:
wid = 'ws4-div'
@app.ws('/ws4', conn=ws_conn(wid), disconn=ws_disconn(wid))
async def ws4(wid:str, msg:str, send):
    _msgs.append((wid, msg))
    if msg == '__init__': return
    await send(_note(wid, msg))

In [ ]:
await user_ws_connect(wid, ws_url(f"/ws4?wid={wid}", srv.port))
_ws_sends.keys()

dict_keys(['ws4-div'])

Verify browser → Python and Python → browser on the registry-backed socket:

In [ ]:
await iife_a(f"user_ws.send('{wid}', {{msg: 'registry'}})")
await wait_for(lambda: 'registry' in str(_msgs[-1]))
await ws_send(wid, _note(wid, 'Hello from registry'))

```
{"msg":"registry"}	18	12:03:38.200
{"msg":"__init____"}	18	12:04:06.446
<div id="note2">Hello registry</div>	37	12:04:06.562
<div id="note2">Hello from registry</div>	42	12:04:06.571
```

`#note2` updated.

The registry-backed version is verified in both directions.

Next small refactor: make a helper that both registers the route and returns the initiator id/path pair, so callers don’t have to repeat the id in three places.

In [ ]:
def add_user_ws(app, wid, path, handler):
    app.ws(path, conn=ws_conn(wid), disconn=ws_disconn(wid))(handler)

A tiny test endpoint can use it like this:

In [ ]:
async def ws5(wid:str, msg:str, send):
    _msgs.append((wid, msg))
    if msg == '__init__': return
    await send(_note(wid, msg))

wid, path = 'ws5-div', '/ws5'
add_user_ws(app, wid, path, ws5)

In [ ]:
await user_ws_connect(wid, ws_url(path+f"?wid={wid}", srv.port))
await iife_a(f"user_ws.send('{wid}', {{msg: 'helper'}})")
await wait_for(lambda: 'helper' in str(_msgs[-1]))
await ws_send(wid, _note(wid, 'Hello from helper'))

In [ ]:
async def start_user_ws(srv, wid, path): return await user_ws_connect(wid, ws_url(path+f"?wid={wid}", srv.port))

In [ ]:
wid, path = 'ws6-div', '/ws6'
add_user_ws(app, wid, path, ws5)
await start_user_ws(srv, wid, path)
await iife_a(f"user_ws.send('{wid}', {{msg: 'started'}})")
await wait_for(lambda: 'started' in str(_msgs[-1]))
await ws_send(wid, _note(wid, 'Hello from start_user_ws'))

We now have the pieces separated cleanly:

- `ws_url` builds the public websocket URL
- `user_ws_create` creates/processes the browser initiator
- `_ws_sends` tracks active Python send callables by initiator id
- `ws_conn` / `ws_disconn` manage that registry
- `ws_send` sends Python → browser
- `add_user_ws` registers an endpoint
- `start_user_ws` starts the browser side

`v1` will package these into a small reusable section, probably with the JS string plus these Python helpers together.

In [ ]:
await user_ws_cleanup('ws4-div')
await user_ws_cleanup('ws5-div')
await user_ws_cleanup('ws6-div');

## version 1

Let's consolidate the Python-facing helper layer first:

In [ ]:
def user_ws_url(path, port=None, qs:dict=None, sch='wss'):
    "Build a Solveit public websocket URL for the given port/path/query params: <sch>://<public domain for port>/<path>?<qs>"
    return f"{sch}://{pub_domain(port if port is not None else free_port(public=True))}/{path.lstrip('/')}{'?' if qs else ''}{urlencode(qs or {})}"

In [ ]:
_pub_ports = pub_ports()

user_ws_url('foo/bar', _pub_ports[-1], dict(abc='a/b/c'))

'wss://youthful-spiral-carries-nri2kn.solveit.pub/foo/bar?abc=a%2Fb%2Fc'

In [ ]:
async def user_ws_call(fn, *args):
    args = ', '.join(json.dumps(o) for o in args)
    return await js_eval_a(f"return user_ws.{fn}({args})")

async def user_ws_send(wid, data):
    l = len(_msgs)
    await iife_a(f"user_ws.send({json.dumps(wid)}, {json.dumps(data)})")
    await wait_for(lambda: len(_msgs)>l)

In [ ]:
wid,path = 'ws7-div', '/ws7'
add_user_ws(app, wid, path, ws5)
await user_ws_connect(wid, user_ws_url(path, srv.port, dict(wid=wid)))
await user_ws_send(wid, dict(msg='packed'))
test_is('packed' in str(_msgs[-1]), True)
await ws_send(wid, _note(wid, 'Hello from packed helpers'))

In [ ]:
await user_ws_cleanup(wid)

Consolidate the server-side registry helpers into the same reusable shape, with a clearer public name:

In [ ]:
user_ws_sends = {}

def user_ws_conn(wid):
    async def _conn(send): user_ws_sends[wid] = send
    return _conn

def user_ws_disconn(wid):
    async def _disconn(ws): user_ws_sends.pop(wid, None)
    return _disconn

async def user_ws_send_py(wid, o): return await user_ws_sends[wid](o)

def add_user_ws(app, wid, path, handler):
    app.ws(path, conn=user_ws_conn(wid), disconn=user_ws_disconn(wid))(handler)

async def user_ws_connect(wid, wsurl):
    await user_ws_create_a(wid, wsurl)
    await wait_for(lambda: wid in user_ws_sends)
    await wait_for(lambda: (wid, '__init__') in _msgs)

In [ ]:
wid,path = 'ws8-div', '/ws8'
add_user_ws(app, wid, path, ws5)
await user_ws_connect(wid, user_ws_url(path, srv.port, dict(wid=wid)))
await user_ws_send(wid, dict(msg='renamed'))
await user_ws_send_py(wid, _note(wid, 'Hello from renamed registry'))
await user_ws_cleanup(wid)

Make a single convenience function that registers the route and starts the browser initiator.

In [ ]:
async def start_user_ws(app, srv, wid, path, handler):
    add_user_ws(app, wid, path, handler)
    return await user_ws_connect(wid, user_ws_url(path, srv.port, dict(wid=wid)))

In [ ]:
wid, path = 'ws9-div', '/ws9'
await start_user_ws(app, srv, wid, path, ws5)
await user_ws_send(wid, dict(msg='onecall'))
test_is('onecall' in str(_msgs[-1]), True)
await user_ws_send_py(wid, _note(wid, 'Hello from start_user_ws'))

Make shutdown symmetric so intentional shutdown removes the initiator first and then drops the stored send callable.

In [ ]:
async def stop_user_ws(id):
    try: await user_ws_cleanup(id)
    finally: user_ws_sends.pop(id, None)
    assert wid not in user_ws_sends

In [ ]:
await stop_user_ws(wid)

That's a lot and a lot of `user_ws_` prefixes.  Time to put in a class.

This is the right point to put the public surface behind one object. The class can own `app`, `srv`, JS namespace, and the active send registry.

In [ ]:
class UserWS:
    def __init__(self, app, srv, js='user_ws'): store_attr(); self.sends, self._msgs = {}, []

    def url(self, path, qs:dict=None, sch='wss'):
        return f"{sch}://{pub_domain(self.srv.port)}/{path.lstrip('/')}{'?' if qs else ''}{urlencode(qs or {})}"

    async def create(self, wid, path):
        wsurl = path if path.startswith('ws') else self.url(path, dict(wid=wid))
        await add_html_a((
            Div(hx_swap_oob='afterend:#ws-div')(Div(ws_connect=wsurl, hx_ext='ws', id=wid, data_uws=True)),
            Div(hx_swap_oob='beforebegin:#uws-note')(_indicator(wid))))

    async def cleanup(self, wid):
        res = await js_eval_a(f'''setTimeout(() => {{ $("#{wid}-uws-indicator").remove() }}, 500); return user_ws.cleanup({json.dumps(wid)})''')
        assert res.get('result') in ('removed', 'not found')
    async def send_client(self, wid, data):
        l = len(self._msgs)
        await iife_a(f"{self.js}.send({json.dumps(wid)}, {json.dumps(data)})")
        await wait_for(lambda: len(self._msgs)>l)
    async def send(self, wid, o):
        assert wid in self.sends, f"{wid} is not connected"
        return await self.sends[wid](o)

    def conn(self, wid):
        async def _conn(send): self.sends[wid] = send
        return _conn
    def disconn(self, wid):
        async def _disconn(ws): self.sends.pop(wid, None)
        return _disconn

    def add(self, path, handler, wid=''):
        wid = wid or _new_id('uws')
        self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
        return wid
    async def connect(self, wid, path):
        await self.create(wid, path)
        await wait_for(lambda: wid in self.sends)
        await wait_for(lambda: (wid, '__init__') in self._msgs)
    async def start(self, path, handler, wid=''):
        wid = self.add(path, handler, wid=wid)
        await self.connect(wid, path)
        return wid
    async def restart(self, path, handler, wid): return await self.start(path, handler, wid=wid)

    async def stop(self, wid):
        try: await self.cleanup(wid)
        finally: self.sends.pop(wid, None)
        assert wid not in self.sends

In [ ]:
uws = UserWS(app, srv)
_msgs = uws._msgs

In [ ]:
wid = await uws.start(path := '/ws12', ws5)
wid

'uws_1'

In [ ]:
await uws.send_client(wid, dict(msg='classy'))
test_is('classy' in str(_msgs[-1]), True)
await uws.send(wid, _note(wid, 'Hello from UserWS'))
uws._msgs

[('uws_1', '__init__'), ('uws_1', 'classy')]

The class shape is now verified:

- `uws.start(...)` registers the endpoint, creates/processes the browser initiator, and waits for connection
- `uws.send_client(...)` sends browser → Python
- `uws.send(...)` sends Python → browser
- `uws.sends` cleanly scopes active send callables per `UserWS` instance

Next: make `start` return `wid` as before, but first clean up any existing initiator with the same id to avoid duplicate DOM elements/retry loops during iteration.

In [ ]:
@patch
async def start(self:UserWS, path, handler, wid=''):
    if wid: await self.stop(wid)
    wid = self.add(path, handler, wid=wid)
    await self.connect(wid, path)
    return wid

One thing to be careful about: `FastHTML.app.ws(...)` may not like re-registering the same route path during iteration. Cleaning up the DOM initiator is good, but `self.add(...)` still registers the route again.

So next tiny refinement: split “route already added” from “browser initiator started” by tracking registered websocket ids.

In [ ]:
@patch
def _init_routes(self:UserWS):
    if not hasattr(self, 'routes'): self.routes = set()

@patch
def add(self:UserWS, path, handler, wid=''):
    self._init_routes()
    wid = wid or _new_id('uws')
    if wid not in self.routes:
        self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
        self.routes.add(wid)
    return wid

In [ ]:
await uws.start(path, ws5, wid)
test_eq((wid in uws.sends, wid in uws.routes), (True, True))

In [ ]:
await uws.send_client(wid, dict(msg='restarted'))
await uws.send(wid, _note(wid, 'Hello after restart'))
await uws.stop(wid)

Nice. One more small polish: initialize `routes` in `__init__` so `_init_routes` isn't needed in the final class.

In [ ]:
@patch
def __init__(self:UserWS, app, srv, js='user_ws'):
    store_attr()
    self.sends,self.routes,self._msgs = {},set(),[]

@patch
def add(self:UserWS, path, handler, wid=''):
    wid = wid or _new_id('uws')
    if wid not in self.routes:
        self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
        self.routes.add(wid)
    return wid

In [ ]:
uws2 = UserWS(app, srv)
_msgs = uws2._msgs
wid2 = await uws2.start(path2 := '/ws13', ws5, wid)
test_eq((wid2 == wid, wid2 in uws2.sends, wid2 in uws2.routes), (True, True, True))

In [ ]:
await uws2.send_client(wid2, dict(msg='fresh_instance'))
test_is('fresh_instance' in str(_msgs[-1]), True)
await uws2.send(wid2, _note(wid, 'Hello from fresh instance'))

Next small tightening: `routes` should probably be a dict from `wid → path`, not a set. That makes restart idempotent for the same `wid/path`, but catches accidental reuse of the same `wid` for a different route.

In [ ]:
@patch
def __init__(self:UserWS, app, srv, js='user_ws'):
    store_attr()
    self.sends,self.routes,self._msgs = {},{},[]

@patch
def add(self:UserWS, path, handler, wid=''):
    wid = wid or _new_id('uws')
    if wid in self.routes:
        assert self.routes[wid] == path, f"{wid} already registered for {self.routes[wid]}, not {path}"
        return wid
    self.app.ws(path, conn=self.conn(wid), disconn=self.disconn(wid))(handler)
    self.routes[wid] = path
    return wid

Test for the duplicate-`wid` guard:

In [ ]:
uws3 = UserWS(app, srv)
_msgs = uws3._msgs
wid3 = uws3.add('/ws14', ws5, 'same-wid')
test_eq(uws3.add('/ws14', ws5, 'same-wid'), wid3)
test_fail(lambda: uws3.add('/ws15', ws5, 'same-wid'), contains='already registered')

In [ ]:
wid4 = await uws3.start('/ws14', ws5, 'same-wid')
res = await uws3.stop(wid4)
test_is(wid4 in uws3.sends, False)
await uws3.stop('missing-wid')

In [ ]:
await uws2.stop(wid2)
await stop_server_a()

## version 2

The class works from a fresh instance.

Next, let's prepare the final consolidated JS string and class. First the JS string:

In [ ]:
await add_js_msg('_user_ws_js', i_collapsed=True, exported=True);

In [ ]:
_user_ws_js = "if (window.user_ws) {\n  window.user_ws.listen(false);\n  delete window.user_ws;\n}\n\nfunction hooks() { return {open:[], close:[], err:[], cleanup:[]} }\nconst _hooks = hooks()\nfunction on(name, fn) { _hooks[name].push(fn); return () => off(name, fn); }\nfunction off(name, fn) { _hooks[name] = _hooks[name].filter(o => o !== fn) }\nfunction call(name, ...args) { for (const fn of _hooks[name]) fn(...args); }\n\nfunction _elt(e) { return e.detail?.elt || e.target }\nfunction _onopen(e) {\n  const elt = _elt(e);\n  if (!elt?.hasAttribute('data-uws')) return;\n  e.stopImmediatePropagation();\n  elt._ws = e.detail.socketWrapper;\n  setTimeout(() => send(elt.id, {event:'__init__'}), 250);\n  call('open', elt, e);\n}\nfunction _onerr(e) {\n  const elt = _elt(e);\n  if (!elt?.hasAttribute('data-uws')) return;\n  e.stopImmediatePropagation();\n  call(e.type === 'htmx:wsClose' ? 'close' : 'err', elt, e)\n}\nasync function _opened(e) {\n  const {idx,wid,timeout=3000} = e.detail;\n  const end = Date.now()+timeout;\n  while (Date.now()<end) {\n    const el = document.getElementById(wid);\n    if (el?._ws) return pushData(idx, {open:true});\n    await new Promise(r => setTimeout(r, 25));\n  }\n  pushData(idx, {open:false});\n}\nfunction listen(on=true) {\n  const l = (on ? document.addEventListener : document.removeEventListener).bind(document);\n  l('htmx:wsOpen', _onopen, true);\n  l('htmx:wsClose', _onerr, true);\n  l('htmx:wsError', _onerr, true);\n  l('uws:open?', _opened, false);\n}\nfunction cleanup(id) {\n  const el = document.getElementById(id);\n  if (!el) return 'not found';\n  htmx.remove(el);\n  call('cleanup', el);\n  return 'removed';\n}\nfunction send(id, data={}) {\n  const el = document.getElementById(id);\n  if (!el?._ws) return;\n  const o = typeof data === 'string' ? {msg:data} : {...data};\n  o.wid ??= id;\n  el._ws.send(JSON.stringify(o), el);\n}\n\nlisten();\nconst doc = 'Browser helper for Solveit-safe HTMX websocket initiators';\nconst user_ws = {doc, listen, cleanup, send, on, off};\nwindow.user_ws = user_ws\nexport { user_ws }"
#| linkedto: 84d6a28f

In [ ]:
await add_js_msg('_user_ws_indicator_js', i_collapsed=True, exported=True);

In [ ]:
_user_ws_indicator_js = "const user_ws = window.user_ws;\n\nfunction truncS(s, l=120) { s = String(s); return s.length <= l ? s : s.substring(0, l) + '\\u2026'; }\nfunction shownote(id, msg, dir, color) {\n  $('#uws-note').empty()\n    .append($('<span>').css('color', color).text(dir))\n    .append(document.createTextNode(`${id}: `))\n    .append($('<samp>').text(truncS(msg)))\n}\nfunction noteup(e) { shownote(e.detail.elt.id, e.detail.message, '⬆︎', 'green') }\nfunction notedown(e) { shownote(e.detail.elt.id, e.detail.message, '⬇︎', 'red') }\nfunction onopen(el,e) {\n  $(`#${el.id}-uws-indicator`).css('backgroundColor', 'limegreen');\n  htmx.on(el, 'htmx:wsAfterMessage', noteup);\n  htmx.on(el, 'htmx:wsAfterSend', notedown);\n  shownote(el.id, 'opened', '⬇︎', 'red')\n}\nfunction onerr(el,e) {\n  const type = e.detail?.event?.type || 'error';\n  const color = { error: 'orange', close: 'red' }[type] || 'grey';\n  $(`#${el.id}-uws-indicator`).css('backgroundColor', color);\n}\nfunction oncleanup(el) {\n  $(`#${el.id}-uws-indicator`).css('backgroundColor', 'grey');\n  htmx.off(el, 'htmx:wsAfterMessage', noteup);\n  htmx.off(el, 'htmx:wsAfterSend', notedown);\n  shownote(el.id, 'closed', '⬇︎', 'red')\n}\n\nif (window._uws_indicator_offs) window._uws_indicator_offs.forEach(f => f());\nwindow._uws_indicator_offs = [\n  user_ws.on('open', onopen),\n  user_ws.on('close', onerr),\n  user_ws.on('err', onerr),\n  user_ws.on('cleanup', oncleanup)\n];"
#| linkedto: c049acaa

In [ ]:
test_eq(await hx.remove('#uws-note'), True)

In [ ]:
#| exporti
async def setup_uws_indicator(dname:str=''):
    await iife_a(r"document.getElementById('uws-box')?.remove()")
    await add_html_a(
        Div(hx_swap_oob='beforebegin:#dialog-container')(
            Div(id='uws-box', cls='flex flex-wrap w-full gap-1 lg:flex-nowrap')(
                Div('uws indicator...', id='uws-note')
            )))
    await setup_mod('user_ws_indicator', _user_ws_indicator_js, dname=dname)

In [ ]:
#| export
async def setup_user_ws(indicator=False, dname:str=''):
    "Inject the browser-side user_ws helper; use only in Solveit environment"
    await setup_mod('user_ws', _user_ws_js, dname=dname)
    if indicator: await setup_uws_indicator(dname=dname)

In [ ]:
await setup_user_ws(True)

The browser helper in `_user_ws_js` is injected once with `setup_user_ws()`, while `UserWS(app, srv)` instances can be created many times to manage independent route/send registries.

The JS helper tracks websocket initiator elements, stores each HTMX socket wrapper on its DOM element as `el._ws`, and filters HTMX websocket events by `data-uws` attribute. Matching `htmx:wsOpen`, `htmx:wsClose`, and `htmx:wsError` events are stopped in the capture phase so Solveit’s own global websocket monitor does not mistake user websocket lifecycle events for Solveit backend failures.

In [ ]:
#| exporti
async def noop_a(*args, **kwargs): pass

async def create_uws_indicator(wid):
    return await add_html_a(Div(hx_swap_oob='beforebegin:#uws-note')(_indicator(wid)))

async def cleanup_uws_indicator(wid):
    return await js_eval_a(f'''return $("#{wid}-uws-indicator").remove().length > 0''')

uws_ind = dict(create_ui=create_uws_indicator, cleanup_ui=cleanup_uws_indicator)

In [ ]:
#| exporti
def user_ws_url(path, port:int, qs:dict=None, sch='wss'):
    "Build a Solveit public websocket URL for the given port/path/query params: <sch>://<public domain for port>/<path>?<qs>"
    return f"{sch}://{pub_domain(port)}/{path.lstrip('/')}{'?' if qs else ''}{urlencode(qs or {})}"

_JS='user_ws'

def _wid(wid): return wid or _new_id('uws')

async def _wait_open(wid, timeout=3000):
    res = await event_get_a('uws:open?', wid=wid, timeout=timeout)
    assert res.get('open'), f"{wid} did not open"

In [ ]:
#| exporti
def _url(url, wid, port):
    u = urlparse(url)
    qs = dict(parse_qsl(u.query, keep_blank_values=True))
    if not u.netloc: u = urlparse(user_ws_url(u.path, port, qs, sch=u.scheme or 'wss'))
    return urlunparse(u._replace(query=urlencode(dict(qs, wid=wid)))),u.path

`_url(url, wid, port)` normalizes a websocket URL and ensures it carries the initiator ID as a `wid` query parameter.

It returns:

```python
(full_ws_url, route_path)
```

In [ ]:
_url('/ws?abc=ABC', 'uws23', 8001)

('wss://zen-koala-sings-pww6ud.solveit.pub/ws?abc=ABC&wid=uws23', '/ws')

An already-absolute URL keeps its host and path:

In [ ]:
_url('wss://test.solveit.pub/ws2?abc=ABC&wid=old', 'uws23', 8001)

('wss://test.solveit.pub/ws2?abc=ABC&wid=uws23', '/ws2')

So its practical purpose is to let callers accept either a short local route or a complete websocket URL, while consistently passing the websocket initiator identity to the server.

In [ ]:
_url('/ws', 'uws23', _pub_ports[-1])

('wss://youthful-spiral-carries-nri2kn.solveit.pub/ws?wid=uws23', '/ws')

In [ ]:
_url('ws', 'uws23', _pub_ports[-1])

('wss://youthful-spiral-carries-nri2kn.solveit.pub/ws?wid=uws23', '/ws')

In [ ]:
_url('wss://test.solveit.pub/_droute/ws/cHJqLoP0ADRawT_qyzej2Y/ws2?abc=ABC&wid=w1&def=DEF', 'uws23', _pub_ports[-1])

('wss://test.solveit.pub/_droute/ws/cHJqLoP0ADRawT_qyzej2Y/ws2?abc=ABC&wid=uws23&def=DEF',
 '/_droute/ws/cHJqLoP0ADRawT_qyzej2Y/ws2')

In [ ]:
class UserWS:
    """Manage Solveit-safe HTMX websocket initiators and server send handles keyed by `wid`.
    
    One dialog-local browser websocket initiator per registered route; not shared across concurrent viewers.
    """

    def __init__(self, app, srv, create_ui=noop_a, cleanup_ui=noop_a): store_attr(); self.states,self.paths = {},{}

    async def _create(self, url, wid, **initkw):
        await add_html_a(Div(hx_swap_oob='afterend:#ws-div')(Div(ws_connect=url, hx_ext='ws', id=wid, data_uws=True, **initkw)))
        await self.create_ui(wid)

    async def cleanup(self, wid):
        "Remove browser initiator element"
        try:
            res = await js_eval_a(f'''return {_JS}.cleanup({json.dumps(wid)})''')
            assert res.get('result') in ('removed', 'not found')
        finally: await self.cleanup_ui(wid)
    async def send_client(self, wid, data):
        "Send browser to Python via JS helper"
        await iife_a(f"{_JS}.send({json.dumps(wid)}, {json.dumps(data)})")
    async def send(self, wid, o):
        "Send Python to browser via stored send callable"
        assert (state:=self.states.get(wid)) and state.send, f"{wid} is not connected"
        return await state.send(o)

    def _conn(self, wid, conn):
        async def _conn(scope,ws,send):
            state = self.states[wid]
            state.ws,state.send = ws,send
            if conn: await conn(scope,ws,send)
        return _conn
    def _disconn(self, wid, disconn):
        async def _disconn(scope,ws):
            if (state:=self.states.get(wid)) and state.ws is ws: state.ws,state.send = None,None
            if disconn: await disconn(scope,ws)
        return _disconn

    @delegates(FastHTML.ws)
    def add(self, wid, handler, path, conn=None, disconn=None, **kwargs) -> str: # `wid`
        "Register websocket route once; optionally setup conn/disconn hooks"
        if wid in self.states:
            assert self.states[wid].path == path, f"{wid} already registered for {self.states[wid].path}, not {path}"
            return wid
        if path in self.paths: raise AssertionError(f"{path} already registered for {self.paths[path]}, not {wid}")
        self.app.ws(path, conn=self._conn(wid, conn), disconn=self._disconn(wid, disconn), **kwargs)(handler)
        self.states[wid],self.paths[path] = AD(path=path, ws=None, send=None),wid
        return wid
    
    @delegates(add, keep=True)
    async def start(self,
        url, # Absolute `ws://`/`wss://` URL or route path; relative paths use this instance's public server URL
        handler, # FastHTML websocket handler; receives the JSON fields sent by the browser, including `wid`
        wid='', # Browser initiator id; generated when omitted; an existing initiator with this id is stopped first
        path=None, # Route path; defaults to the URL path
        initkw:dict=None, # Additional attributes for the browser websocket initiator element
        **kwargs
    )->str: # Connected websocket initiator id (`wid`)
        "Register one websocket route, create its Solveit-safe browser initiator, and wait until browser and Python can both send."
        if wid: await self.stop(wid)
        else: wid = _wid(wid)
        url,route_path = _url(url, wid, self.srv.port)
        path = route_path if path is None else path
        try:
            self.add(wid, handler, path, **kwargs)
            await self._create(url, wid, **(initkw or {}))
            await _wait_open(wid)
            await wait_for(lambda: self.states[wid].send is not None)
        except BaseException:
            await self.stop(wid)
            raise
        return wid

    async def stop(self, wid):
        "Remove browser initiator and forget server send callable"
        try: await self.cleanup(wid)
        finally:
            if state := self.states.get(wid): state.ws,state.send = None,None

In [ ]:
_msgs,wid = [], None
Div(id=(did := f"_{simple_id()}"))

<div id="_4d06a520"></div>

In [ ]:
srv, app, rt, p = await ensure_server_a(app=FastHTML(exts='ws', debug=True), public=True)

async def ws_msg(data:dict, ws, send):
    _msgs.append(data)
    if data.get('event') == '__init__': return
    await send(Div(Samp(json.dumps(data)), Br(), id=did, hx_swap_oob="beforeend"))

# async with serve_app(app_kw=AD(exts='ws')) as srv:#, log_level='trace') as srv:
uws3 = UserWS(srv.app, srv, **uws_ind)    
wid = await uws3.start(path := '/ws14', ws_msg)

print(_msgs, '\n----\n')

await uws3.send_client(wid, dict(msg='clean'))
await wait_for(lambda: _msgs and 'clean' in str(_msgs[-1]))
await uws3.send(wid, Div(Samp('Hello from cleaned UserWS'), Br(), id=did, hx_swap_oob="beforeend"))

await uws3.stop(wid)
print(_msgs)

[] 
----


[{'event': '__init__', 'wid': 'uws_2'}, {'msg': 'clean', 'wid': 'uws_2'}]


In [ ]:
await stop_server_a()

###### start-section

Critique `UserWS` in light of the narrative of this dialog. Note that I've changed the startup handshake and handler signature. No need to compare with existing `pote.user_ws` pyskill (it'll be generated below). This is a minor refactoring.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

This is a good minor refactor. It aligns well with the dialog’s conclusions:

- the browser helper owns HTMX event isolation and socket wrappers;
- `UserWS` owns route registration, browser initiators, and Python send handles;
- `wid` is now carried in every browser message, so handlers receive a structured `data: dict` rather than a special positional `msg`;
- `wait_open()` plus `wid in self.sends` is a cleaner readiness check than treating `__init__` receipt as the startup contract.

The main points I would address before treating this as settled:

1. **Restart can remove its newly-created indicator.**  
   `start(..., wid=...)` calls `stop(wid)`, whose `cleanup_ui` currently schedules indicator removal after 500 ms. The new initiator and indicator are then created immediately with the same id, so that delayed callback can remove the *new* indicator. Cleanup should remove the old indicator immediately, or identify the specific old element rather than selecting solely by `wid`.

2. **A failed `start` leaves the initiator alive.**  
   If `wait_open()` or the server-send wait times out, the browser initiator remains in the DOM and HTMX may continue reconnecting. This is precisely the case where the earlier narrative recommends removal. Wrap creation/readiness in `try`/`except`, call `stop(wid)` on failure, then re-raise.

3. **Existing routes silently retain their original handler and hooks.**  
   When a `wid` is already registered, `add` checks only the path and returns. A later `start(path, new_handler, wid=...)` restarts the browser connection but does **not** install `new_handler`, `conn`, or `disconn`. In an iterative notebook this is easy to miss after redefining a handler. Either document that routes are immutable for an instance lifetime, or make the duplicate-route case explicit in the API name/exception.

4. **The class currently means one `wid` per route path.**  
   `self.paths` rejects a second `wid` for the same path. That is coherent if the intended contract is “one browser initiator per endpoint,” but it narrows the earlier registry motivation: several initiators cannot share one FastHTML websocket route. Supporting that later would require deriving `wid` from the connection scope/query string rather than closing over it in `_conn`.

5. **`_url` should use proper query parsing.**  
   The current `dict(_.split('=') for _ in u.query.split('&'))` breaks for encoded or literal `=` characters, drops repeated query keys, and does not properly decode values. `parse_qsl(..., keep_blank_values=True)` is the standard safe replacement.

The new handshake itself is sound. Browser readiness is established by `uws:open?`; server readiness is established by the stored `send` callable; the later `{event: '__init__', wid}` message is now a normal structured lifecycle message for the handler to ignore or record, rather than a special mechanism required to prove startup.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 51358, "completion_tokens": 2991, "total_tokens": 54349, "cached_tokens": 3456, "reasoning_tokens": 2344, "cost": 0.00994695}
```


Ok, let's address each issue in turn.

5. [x] **`_url` should use proper query parsing.** -> fixed

1. [ ] **Restart can remove its newly-created indicator.**

Not clear how to solve this or even if it's really an issue. What do you propose?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

It is a real race in the current code:

1. `start(..., wid='x')` calls `stop('x')`.
2. `cleanup_uws_indicator('x')` schedules `$('#x-uws-indicator').remove()` for 500 ms later.
3. `start` creates a new `x-uws-indicator` immediately.
4. The old timer fires and may remove the new indicator.

The simplest fix is to make indicator cleanup synchronous. Browser initiator cleanup already happens first, so there is no need to defer removal of its separate indicator.

Replace `cleanup_uws_indicator` with:

```python
async def cleanup_uws_indicator(wid):
    return await js_eval_a(f'''return $("#{wid}-uws-indicator").remove().length > 0''')
```

That establishes the useful ordering: remove old initiator → remove old indicator → create new initiator and indicator.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 52227, "completion_tokens": 486, "total_tokens": 52713, "cached_tokens": 8576, "reasoning_tokens": 287, "cost": 0.00819855}
```


Hmmm, don't remember why I setup the timeout originally (but surely for a reason, I'm not in the habit of setting up timeouts), but anyhow, fixed.
1. [x] **Restart can remove its newly-created indicator.**

---

2. [ ] **A failed start leaves the initiator alive.**


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`start` should treat creation and readiness as one transaction: if either readiness check fails, remove the initiator before re-raising the original failure.

```python
@delegates(UserWS.add, keep=True)
async def start(self:UserWS, url, handler, wid='', initkw:dict=None, **kwargs):
    if wid: await self.stop(wid)
    else: wid = _wid()
    url,path = _url(url, wid, self.srv.port)
    try:
        self.add(wid, handler, path, **kwargs)
        await self._create(url, wid, **(initkw or {}))
        await self.wait_open(wid)
        await wait_for_a(lambda: wid in self.sends)
    except BaseException:
        await self.stop(wid)
        raise
    return wid
```

The registered FastHTML route remains after failure; that is useful, since a later retry with the same `wid` and `path` reuses it. The browser initiator, its indicator, and any stale stored `send` handle are removed.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 52538, "completion_tokens": 689, "total_tokens": 53227, "cached_tokens": 8576, "reasoning_tokens": 460, "cost": 0.00839745}
```


2. [x] **A failed start leaves the initiator alive.** -> Fixed.

---

4. [ ] **The class currently means one `wid` per route path.**

`UserWS.add` is in flux. I'm not even convinced is the right shape/contract. What would be the use case for several initiators sharing the same path?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Several initiators sharing a path are useful when the **path names a protocol/service**, while `wid` names a particular browser-side connection.

Examples:

- A chat endpoint at `/chat`, with one initiator per open conversation or pane.
- Several live widgets on one page, all speaking the same message schema at `/updates`.
- Two browser tabs connected to the same route.
- A reconnect/replacement connection where the route stays constant but each client instance has its own `wid`.
- A broadcast service: one `/events` route, many recipients, each with its own stored `send` callable.

In those cases, the server handler uses `data['wid']` to identify the sender, and the send registry remains keyed by `wid`. The route should be registered once by `path`, not once by `wid`.

But none of those are required by the current narrative. So far, each test has created one route and one initiator together. A deliberate narrower contract is reasonable:

> A `UserWS` instance manages one initiator per websocket route path.

That makes `routes: wid → path` and `paths: path → wid` valid, prevents accidental duplicate initiators, and keeps connection hooks simple because they can close over `wid`.

The API should therefore make that restriction explicit rather than quietly implying a general websocket multiplexer. `add` is slightly ambiguous because it registers a route but also binds it permanently to an initiator identity. A name such as `add_initiator` or `register` would make the contract clearer.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 52916, "completion_tokens": 593, "total_tokens": 53509, "cached_tokens": 52608, "reasoning_tokens": 274, "cost": 0.008382150000000001}
```


All that opens a can of worms. In paticular, Solveit contract is two browsers can open the same URL and see exactly the same state. One change in one browser is reflected automatically in the other. With UserWS that's not the caseat all because the initiator is injected dinamically (unlike the Solveit initiator that is part of the initial page).

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. That makes the multi-initiator case largely beside the point.

`UserWS` is not a general shared websocket service analogous to Solveit’s built-in connection. It is a **dialog-local, dynamically injected browser capability**:

- `UserWS.start()` adds an initiator element to the current browser’s DOM.
- Its `wid`, socket wrapper, indicator, and lifecycle hooks exist only in that browser.
- Another browser viewing the same dialog has neither that injected element nor its corresponding connection.
- The Python kernel may be shared, but the browser-side websocket state is not.

So a single route shared by several `wid`s would imply a distributed UI/state contract that this abstraction neither implements nor should quietly suggest. It would raise questions about who receives updates, how new viewers initialize, whether stopped initiators are re-created, and how browser-specific state is reconciled.

The narrow contract is appropriate:

> `UserWS` manages one dynamically injected websocket initiator for one route in the active browser context.

Under that contract, rejecting a second `wid` for an existing `path` is not an incidental limitation; it prevents callers from assuming shared-client semantics that are unavailable. `routes` and `paths` are therefore useful guardrails.

I would describe this explicitly in the class docstring or `add`/`start` docments, rather than expand the API:

```python
"Manage one dialog-local browser websocket initiator per registered route; not shared across concurrent viewers"
```

That keeps `wid → path` and `path → wid` one-to-one, retains simple connection hooks, and avoids opening the multi-browser synchronization problem.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 53366, "completion_tokens": 523, "total_tokens": 53889, "cached_tokens": 52608, "reasoning_tokens": 184, "cost": 0.008397149999999999}
```


The only way to (begin) solving that is persisting the initiator in a message (input or output, like Solveit does). Feasible, but for a later time.

4. [X] **The class currently means one `wid` per route path.** -> Added note to class docstring.

---

3. [ ] **Existing routes silently retain their original handler and hooks.**

Don't understand that.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

It concerns repeated calls to `start` after redefining a handler in the notebook.

Suppose this has already run:

```python
async def handler_v1(data, ws, send): ...

await uws.start('/ws', handler_v1, wid='my-ws')
```

Later, you redefine it:

```python
async def handler_v2(data, ws, send): ...

await uws.start('/ws', handler_v2, wid='my-ws')
```

`start` stops and recreates the **browser initiator**, but then calls `add`. Since `'my-ws'` is already in `self.routes`, `add` returns early. It does not call `self.app.ws(...)(handler_v2)` again.

Therefore the FastHTML route is still bound to `handler_v1`; `handler_v2` is silently ignored. The same applies to changed `conn` and `disconn` hooks.

This is not necessarily a defect. FastHTML routes are naturally persistent for the server/app lifetime, and attempting to replace them during notebook iteration may not be supported. The concern is only that this call shape suggests otherwise:

```python
await uws.start('/ws', handler_v2, wid='my-ws')
```

A clean contract would distinguish the two actions:

- `add(...)`: register an immutable route, handler, and hooks once.
- `start(...)`: create/restart the browser initiator for an already-registered route.

Then `start` should take `wid` (or perhaps `url`) but **not** `handler`, since it cannot reliably change it.

Alternatively, retain the current combined `start` API but record the original handler/hook objects and raise a clear error if later calls supply different ones.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 53863, "completion_tokens": 646, "total_tokens": 54509, "cached_tokens": 3456, "reasoning_tokens": 281, "cost": 0.00856395}
```


FastHTML **does** support changing dinamically the handler (as Starlette), but I see that's mainly useful during developemnt in a notebook environment. We could support that easily but no need now. Maybe in the future when feeling the pain.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Agreed. The current one-time route-registration contract is sufficient. Dynamic handler replacement is a development convenience, not required for the browser lifecycle abstraction itself.

Record it as an intentional limitation for now: restarting an existing `wid` recreates its browser initiator but retains the route’s originally registered handler and hooks.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 54330, "completion_tokens": 102, "total_tokens": 54432, "cached_tokens": 8576, "reasoning_tokens": 34, "cost": 0.008226}
```


All right, anything else?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

One remaining lifecycle race is worth fixing: an **old socket’s delayed disconnect hook can erase a new socket’s send handle** during restart.

Sequence:

1. `start(wid=...)` removes the old initiator.
2. The new initiator connects and stores its `send`.
3. The old socket finally fires `disconn`.
4. Current `_disconn` unconditionally pops `self.sends[wid]`, removing the *new* handle.

Keep the connection object alongside the send callable, and clear them only if the disconnect belongs to the currently registered connection:

```python
@patch
def __init__(self:UserWS, app, srv, create_ui=noop_a, cleanup_ui=noop_a):
    store_attr()
    self.sends,self.conns,self.routes,self.paths = {},{},{},{}

@patch
def _conn(self:UserWS, wid, conn):
    async def _conn(scope,ws,send):
        self.sends[wid],self.conns[wid] = send,ws
        if conn: await conn(scope,ws,send)
    return _conn

@patch
def _disconn(self:UserWS, wid, disconn):
    async def _disconn(scope,ws):
        if self.conns.get(wid) is ws:
            self.sends.pop(wid, None)
            self.conns.pop(wid, None)
        if disconn: await disconn(scope,ws)
    return _disconn
```

Then make `stop` clear both local records:

```python
@patch
async def stop(self:UserWS, wid):
    try: await self.cleanup(wid)
    finally:
        self.sends.pop(wid, None)
        self.conns.pop(wid, None)
```

Other remaining points are documentation-level: `wid` is an initiator identifier, not an authentication boundary; and `send_client` intentionally does nothing if the browser socket is not currently open.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 54457, "completion_tokens": 1965, "total_tokens": 56422, "cached_tokens": 53632, "reasoning_tokens": 1552, "cost": 0.0096423}
```


That's screaming of refactoring the three maps keyed by wid in a class (or at least a named tuple).

Where did we say that `wid` is related in any way to authentication?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

We never said that `wid` participates in authentication. I introduced that caveat without basis in this design; it is irrelevant here.

And yes: `sends`, connection objects, and registered routes are all properties of the same per-`wid` record. Put them behind one small state object; retain only `paths` as the inverse lookup needed to enforce the one-to-one route contract.

```python
class _UWSState:
    def __init__(self, path): self.path,self.ws,self.send = path,None,None
```

Then `UserWS.__init__` becomes:

```python
self.states,self.paths = {},{}
```

Each `wid` maps to one `_UWSState`. Route registration creates `self.states[wid]`; connection/disconnection updates its `ws` and `send`; `stop` clears the transient fields. `self.paths[path] = wid` remains the guard against a second initiator using the same route.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 54961, "completion_tokens": 628, "total_tokens": 55589, "cached_tokens": 53632, "reasoning_tokens": 426, "cost": 0.008715150000000001}
```


Ok, write this version in full. And remember to respect the docments; don't know why (well, yes, out of distribution) you AIs always forget existing docments when recreating a function.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
class _UWSState:
    def __init__(self, path): self.path,self.ws,self.send = path,None,None

class UserWS:
    """Manage Solveit-safe HTMX websocket initiators and server send handles keyed by `wid`.
    
    One dialog-local browser websocket initiator per registered route; not shared across concurrent viewers.
    """

    def __init__(self, app, srv, create_ui=noop_a, cleanup_ui=noop_a):
        store_attr()
        self.states,self.paths = {},{}

    async def _create(self, url, wid, **initkw):
        await add_html_a(Div(hx_swap_oob='afterend:#ws-div')(Div(ws_connect=url, hx_ext='ws', id=wid, data_uws=True, **initkw)))
        await self.create_ui(wid)

    async def cleanup(self, wid):
        "Remove browser initiator element"
        try:
            res = await js_eval_a(f'''return {_JS}.cleanup({json.dumps(wid)})''')
            assert res.get('result') in ('removed', 'not found')
        finally: await self.cleanup_ui(wid)

    async def send_client(self, wid, data):
        "Send browser to Python via JS helper"
        await iife_a(f"{_JS}.send({json.dumps(wid)}, {json.dumps(data)})")

    async def send(self, wid, o):
        "Send Python to browser via stored send callable"
        state = self.states.get(wid)
        assert state and state.send, f"{wid} is not connected"
        return await state.send(o)

    def _conn(self, wid, conn):
        "Return FastHTML connect hook storing server send callable"
        async def _conn(scope, ws, send):
            state = self.states[wid]
            state.ws,state.send = ws,send
            if conn: await conn(scope, ws, send)
        return _conn

    def _disconn(self, wid, disconn):
        "Return FastHTML disconnect hook clearing server send callable"
        async def _disconn(scope, ws):
            state = self.states.get(wid)
            if state and state.ws is ws: state.ws,state.send = None,None
            if disconn: await disconn(scope, ws)
        return _disconn

    @delegates(FastHTML.ws)
    def add(self, wid, handler, path, conn=None, disconn=None, **kwargs) -> str: # `wid`
        "Register websocket route once; optionally setup conn/disconn hooks"
        if wid in self.states:
            assert self.states[wid].path == path, f"{wid} already registered for {self.states[wid].path}, not {path}"
            return wid
        if path in self.paths: raise AssertionError(f"{path} already registered for {self.paths[path]}, not {wid}")
        self.app.ws(path, conn=self._conn(wid, conn), disconn=self._disconn(wid, disconn), **kwargs)(handler)
        self.states[wid],self.paths[path] = _UWSState(path),wid
        return wid

    @delegates(UserWS.add, keep=True)
    async def start(self,
        url, # Absolute `ws://`/`wss://` URL or route path; relative paths use this instance's public server URL
        handler, # FastHTML websocket handler; receives the JSON fields sent by the browser, including `wid`
        wid='', # Browser initiator id; generated when omitted; an existing initiator with this id is stopped first
        initkw:dict=None, # Additional attributes for the browser websocket initiator element
        **kwargs
    )->str: # Connected websocket initiator id (`wid`)
        "Register one websocket route, create its Solveit-safe browser initiator, and wait until browser and Python can both send."
        if wid: await self.stop(wid)
        else: wid = _wid()
        url,path = _url(url, wid, self.srv.port)
        try:
            self.add(wid, handler, path, **kwargs)
            await self._create(url, wid, **(initkw or {}))
            await _wait_open(wid)
            await wait_for_a(lambda: self.states[wid].send is not None)
        except BaseException:
            await self.stop(wid)
            raise
        return wid

    async def stop(self, wid):
        "Remove browser initiator and forget server send callable"
        try: await self.cleanup(wid)
        finally:
            state = self.states.get(wid)
            if state: state.ws,state.send = None,None
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 55258, "completion_tokens": 2076, "total_tokens": 57334, "cached_tokens": 54656, "reasoning_tokens": 1034, "cost": 0.009845699999999999}
```


###### end-section

In [ ]:
_msgs, events = [], []
async def _conn(scope,ws,send): events.append('connected')
async def _disconn(scope,ws): events.append('disconnected')

async with serve_app(public=True) as srv:
    uws4 = UserWS(srv.app, srv, **uws_ind)
    wid = await uws4.start(path := '/ws15', ws_msg, conn=_conn, disconn=_disconn)
    await uws4.send_client(wid, dict(msg='doc_ok'))
    await wait_for(lambda: _msgs and 'doc_ok' in str(_msgs[-1]))
    await uws4.send(wid, Div(Samp('Hello after doc reinject'), Br(), id=did, hx_swap_oob="beforeend"))
    await uws4.stop(wid)
_msgs, events

([{'event': '__init__', 'wid': 'uws_3'}, {'msg': 'doc_ok', 'wid': 'uws_3'}],
 ['connected', 'disconnected'])

Lifecycle regression test: stopping and restarting the same `wid` must reuse its registered route and restore a live send handle.

In [ ]:
_msgs = []

async with serve_app(public=True) as srv:
    uws = UserWS(srv.app, srv, **uws_ind)
    wid = await uws.start('/ws16', ws_msg, wid='restart-test')
    await uws.stop(wid)
    wid = await uws.start('/ws16', ws_msg, wid=wid)
    test_eq((wid, wid in uws.states, uws.states[wid].send is not None), ('restart-test', True, True))
    await uws.stop(wid)

This pins down the intended contract: stop removes the browser connection and send handle, but preserves route registration for a later restart.

Add the complementary route-collision test: one path must not be registered under two distinct websocket ids.

In [ ]:
async with serve_app(public=True) as srv:
    uws = UserWS(srv.app, srv)
    uws.add('first-wid', ws_msg, '/ws17')
    test_fail(lambda: uws.add('second-wid', ws_msg, '/ws17'), contains='already registered')

`send_client` adds `wid` automatically when callers omit it:

In [ ]:
_msgs = []

async with serve_app(public=True) as srv:
    uws = UserWS(srv.app, srv)
    wid = await uws.start('/ws18', ws_msg, wid='client-wid')
    await uws.send_client(wid, dict(msg='auto-wid'))
    await wait_for(lambda: _msgs and _msgs[-1].get('msg') == 'auto-wid')
    test_eq(_msgs[-1], dict(msg='auto-wid', wid='client-wid'))
    await uws.stop(wid)

In [ ]:
events = []
async def _disconn(scope, ws): events.append('disconnected')

uws = UserWS(None, None)
uws.states['test-wid'] = AD(path=None, ws=None, send=None)

await uws._disconn('test-wid', _disconn)(None, None)

test_eq(events, ['disconnected'])
test_eq(uws.states, {'test-wid': {'path': None, 'ws': None, 'send': None}})

The disconnect hook clears the stored handle and then calls the user hook. Test the symmetric connection contract:

In [ ]:
events = []

async def _conn(scope, ws, send): events.append('connected')
async def send(o): pass

uws = UserWS(None, None)
uws.states['test-wid'] = AD(path=None, ws=None, send=None)
await uws._conn('test-wid', _conn)(None, None, send)

test_eq(events, ['connected'])
test_is(uws.states['test-wid'].send, send)

The remaining non-obvious lifecycle contract is ordering: the stored send handle should already be available inside a user-supplied connection hook.

In [ ]:
seen = []

async def _conn(scope, ws, send): seen.append(uws.states.get('test-wid').send is send)
async def send(o): pass

uws = UserWS(None, None)
uws.states['test-wid'] = AD(path=None, ws=None, send=None)
await uws._conn('test-wid', _conn)(None, None, send)

test_eq(seen, [True])

The connection handle is available before a user connection hook runs. Pin down the reciprocal ordering: it should be removed before a user disconnect hook runs.

In [ ]:
seen = []

async def _disconn(scope, ws): seen.append(uws.states['test-wid'].send is None)

uws = UserWS(None, None)
uws.states['test-wid'] = AD(path=None, ws=None, send=object())
await uws._disconn('test-wid', _disconn)(None, None)

test_eq(seen, [True])

The connection and disconnection ordering contracts are covered.

The hooks system is the remaining gap. It is used by the indicator module, but we have not directly pinned down its public contract.

Start with the normal lifecycle hooks: `open` and `cleanup`.

In [ ]:
await iife_a(r'''
window._uws_events = []
window._uws_offs = [
  user_ws.on('open', el => window._uws_events.push(['open', el.id])),
  user_ws.on('cleanup', el => window._uws_events.push(['cleanup', el.id]))
]
''')

async with serve_app(public=True) as srv:
    uws = UserWS(srv.app, srv)
    wid = await uws.start('/ws-hooks', ws_msg, wid='hook-test')
    await uws.stop(wid)

res = await js_eval_a('return window._uws_events')
test_eq(res['result'], [['open', 'hook-test'], ['cleanup', 'hook-test']])

await iife_a('window._uws_offs.forEach(fn => fn()); delete window._uws_offs')

That verifies registration via `on`, dispatch through `call`, and deregistration using the unsubscribe functions returned by `on`.

`close` and `err` still need an intentional-disconnect test. They matter because those hooks drive the red/orange indicator states; the earlier websocket work established event filtering, but not this hook API boundary directly.

The lifecycle hooks passed: `open` fired when the initiator connected, and `cleanup` fired when `stop` removed it.

Test the unsubscribe function returned by `on`:

In [ ]:
await iife_a(r'''
window._uws_events = []
const off = user_ws.on('open', el => window._uws_events.push(el.id))
off()
''')

async with serve_app(public=True) as srv:
    uws = UserWS(srv.app, srv)
    wid = await uws.start('/ws-hooks-off', ws_msg, wid='hook-off-test')
    await uws.stop(wid)

res = await js_eval_a('return window._uws_events')
test_eq(res['result'], [])

That verifies that a removed hook cannot fire on a subsequent websocket lifecycle event.

The unsubscribe test passed: a hook removed through the function returned by `on(...)` did not fire.

Test `close` and `err` dispatch directly. This isolates the public hook routing without deliberately breaking a live server:

In [ ]:
res = await js_eval_a(r'''
const el = document.createElement('div')
el.id = 'hook-event-test'
el.setAttribute('data-uws', '')
document.body.append(el)

const events = []
const offs = [
  user_ws.on('close', el => events.push(['close', el.id])),
  user_ws.on('err', el => events.push(['err', el.id]))
]

el.dispatchEvent(new CustomEvent('htmx:wsClose', {bubbles:true, detail:{elt:el, event:{type:'close'}}}))
el.dispatchEvent(new CustomEvent('htmx:wsError', {bubbles:true, detail:{elt:el, event:{type:'error'}}}))

offs.forEach(fn => fn())
el.remove()
return events
''')

test_eq(res['result'], [['close', 'hook-event-test'], ['err', 'hook-event-test']])

That verifies that the capture listeners classify `htmx:wsClose` as `close` and `htmx:wsError` as `err`, then pass the websocket initiator element to each registered hook.

The hook-routing test passed: `htmx:wsClose` dispatches `close`, and `htmx:wsError` dispatches `err`, each with the marked initiator element.

The hook system is now covered for:

- registration and dispatch (`open`, `cleanup`)
- unsubscribe functions returned by `on`
- event classification (`close` versus `err`)
- correct initiator identity passed to callbacks

That closes the remaining Version 2 test gap.

Version 2 can now shift from lifecycle internals to a concise public usage example: `setup_user_ws()`, construct `UserWS`, `start`, send in both directions, then `stop`.

In [ ]:
await stop_server_a()
srv, app, rt, _ = await ensure_server_a(public=True)

In [ ]:
test_eq(await hx.remove('#uws-note'), True)

In [ ]:
await setup_user_ws()

In [ ]:
Div(id=(f"uws-reply"))

<div id="uws-reply"></div>

In [ ]:
got = []

async def echo_check(data:dict, ws, send):
    if data.get('event') == '__init__': return
    got.append(data)
    await send(Div(data['msg'], id='uws-reply'))

uws = UserWS(app, srv)

wid = await uws.start('/echo-check', echo_check)
await uws.send_client(wid, 'hello')
await wait_for(lambda: got)
test_eq(got, [dict(msg='hello', wid=wid)])
await uws.stop(wid)

In [ ]:
await stop_server_a()

The end-to-end public example passes against a fresh server:

- `setup_user_ws()` installs the browser helper.
- `UserWS.start()` opens the user websocket.
- `send_client(wid, 'hello')` produces `dict(msg='hello', wid=wid)` in the handler.
- `stop(wid)` cleans up the initiator and connection.

The helper’s core lifecycle and public usage contract are now covered.

The remaining high-value checks are integration checks, not more unit tests:

- In a fresh kernel, run `setup_user_ws()`, create one `UserWS`, then verify `start`, browser→Python, Python→browser, and `stop`.
- Force an unexpected user-websocket disconnect and confirm Solveit’s own connection indicator and submit/cancel controls remain unchanged. That is the original isolation contract.
- Confirm two `UserWS` instances can use distinct routes and `wid`s without their send registries crossing.

The registry and lifecycle-wrapper contracts are already covered directly.

`UserWS` provides Solveit-safe user HTMX websockets. One `UserWS` instance can manage many independent websocket initiators, keyed by `wid`.

Typical flow:

1. `await setup_user_ws()` once per browser session
2. `uws = UserWS(app, srv)`
3. define a FastHTML websocket handler
4. `wid = await uws.start('/ws', handler)`
5. browser → Python: `await uws.send_client(wid, dict(msg='hi'))`
6. Python → browser: `await uws.send(wid, Div('hello', id='target'))`
7. cleanup before stopping the server: `await uws.stop(wid)`

`UserWS.start()` removes any existing browser initiator with the same supplied `wid` before recreating it, avoiding duplicate DOM elements and HTMX retry loops during iteration. Routes are registered only once per `wid` per `UserWS` instance.

Python handlers passed to `UserWS.start()` are normal FastHTML websocket handlers registered with `app.ws(...)`. They take the usual parsed parameters plus special params like `send`, and can send normal FastHTML/HTML responses. `UserWS` only manages connection setup, event isolation, and access to the server-side `send` callable; it does not change FastHTML websocket handler semantics.

### Environment notes

- Solveit's HTTP proxy does not forward websocket upgrades; always use the public `wss://...solveit.pub/...` URL from `UserWS.url()`, not relative paths.
- `on_connect` has a timing race: the socket is open and messages arrive, but HTMX may not yet process the first swap. Treat `conn=` as lifecycle bookkeeping only; send UI updates after an explicit client message.
- The browser helper sends `{event: '__init__'}` 250ms after `htmx:wsOpen` to confirm the connection is settled before the handler sees real messages. Treat it as a best-effort readiness signal and ignore it in normal handlers.

  For relative paths, `UserWS.create()` appends `?wid=...` to the websocket URL. Independently, the browser helper includes `wid` in every JSON message, including the delayed `{event: '__init__'}` marker. Handlers should `return` early on `event == '__init__'`.
- Python → browser websocket responses should be swap-ready HTML. Browser → Python messages are JSON objects sent through `send_client`. Use stable element ids for replacement, or `hx_swap_oob` when appending/inserting into an existing target.
- Reusing a `wid` restarts its browser initiator but does not replace its already-registered FastHTML route, handler, or lifecycle hooks. Use a new `wid` and route path when those need to change.
- Separate `UserWS` instances have independent Python-side registries, but they share the FastHTML app and browser DOM. Their route paths and wid values must not overlap.

**Example**:

```python
await setup_user_ws()
```

```python
async def ws_demo(data:dict, send):
    if data.get('event') == '__init__': return
    await send(Div('Hello ' + data['msg'], id='demo-out'))

async def _conn(scope,ws,send): print('connected')
async def _disconn(scope,ws): print('disconnected')

uws = UserWS(app, srv)
wid = await uws.start('/demo-ws', ws_demo, conn=_conn, disconn=_disconn)
await uws.send_client(wid, dict(msg='world'))

await uws.stop(wid)
```

# pyskill

In [ ]:
import pote.user_ws
from pote.user_ws import setup_user_ws, UserWS

In [ ]:
list_pyskills()['pote.user_ws']

'Solveit compatible user-defined HTMX websockets.'

In [ ]:
doc(pote.user_ws)

# module pote.user_ws:

"""Solveit compatible user-defined HTMX websockets.

The browser helper in `_user_ws_js` is injected once with `setup_user_ws()`, while `UserWS(app, srv)` instances can be created many times to manage independent route/send registries.

The JS helper tracks websocket initiator elements, stores each HTMX socket wrapper on its DOM element as `el._ws`, and filters HTMX websocket events by `data-uws` attribute. Matching `htmx:wsOpen`, `htmx:wsClose`, and `htmx:wsError` events are stopped in the capture phase so Solveit’s own global websocket monitor does not mistake user websocket lifecycle events for Solveit backend failures.

`UserWS` provides Solveit-safe user HTMX websockets. One `UserWS` instance can manage many independent websocket initiators, keyed by `wid`.

Typical flow:

1. `await setup_user_ws()` once per browser session
2. `uws = UserWS(app, srv)`
3. define a FastHTML websocket handler
4. `wid = await uws.start('/ws', handler)`
5. browser → Python: `awa

In [ ]:
doc(setup_user_ws)

async def setup_user_ws(
    indicator:bool=False, dname:str=''
):"""Inject the browser-side user_ws helper; use only in Solveit environment"""

In [ ]:
doc(UserWS)

class UserWS:
    """Manage Solveit-safe HTMX websocket initiators and server send handles keyed by `wid`.
    
    One dialog-local browser websocket initiator per registered route; not shared across concurrent viewers."""
    def __init__(self, app, srv, create_ui=<function noop_a>, cleanup_ui=<function noop_a>): ...
    def add(self, wid, handler, path, conn=None, disconn=None, *, name=None, middleware=None) -> str: ...  # Register websocket route once; optionally setup conn/disconn hooks
    def cleanup(self, wid): ...  # Remove browser initiator element
    def send(self, wid, o): ...  # Send Python to browser via stored send callable
    def send_client(self, wid, data): ...  # Send browser to Python via JS helper
    def start(self, url, handler, wid='', path=None, initkw: dict = None, *, conn=None, disconn=None, name=None, middleware=None, **kwargs) -> str: ...  # Register one websocket route, create its Solveit-safe browser initiator, and wait until browser and Python can both

In [ ]:
doc(UserWS.start)

async def start(
    url, # Absolute `ws://`/`wss://` URL or route path; relative paths use this instance's public server URL
    handler, # FastHTML websocket handler; receives the JSON fields sent by the browser, including `wid`
    wid:str='', # Browser initiator id; generated when omitted; an existing initiator with this id is stopped first
    path:NoneType=None, # Route path; defaults to the URL path
    initkw:dict=None, # Additional attributes for the browser websocket initiator element
    conn:NoneType=None, disconn:NoneType=None, name:NoneType=None, middleware:NoneType=None, **kwargs
)->str: # Connected websocket initiator id (`wid`)"""Register one websocket route, create its Solveit-safe browser initiator, and wait until browser and Python can both send."""

# export -

In [ ]:
#|hide
#|eval: false
from pote.flakes import show_flakes
await show_flakes()

No warnings to report

In [ ]:
#|hide
#|eval: false
from pote.dialog import dlg_export
await dlg_export()

True